In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression  
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

data = pd.read_csv(r'C:\\Users\\JIGNESH\\Desktop\\SSDI P\\cars.csv')
data

data['Age'] = 2025 - data['Year']
data['Transmission'] = data['Transmission'].replace({'Manual': 0, 'Automatic': 1})
data['Fuel_Type'] = data['Fuel_Type'].replace({'Petrol': 0, 'Diesel': 1, 'CNG': 2})
data['Seller_Type'] = data['Seller_Type'].replace({'Dealer': 0, 'Individual': 1})

data.drop(['Car_Name', 'Year'], axis=1, inplace=True)

y = data['Selling_Price']
X = data.drop('Selling_Price', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=123)

LRmodel = LinearRegression()  
LRmodel.fit(X_train, y_train)
print('Train Score:', LRmodel.score(X_train, y_train))
print('Test Score:', LRmodel.score(X_test, y_test))

present_price = float(input("Enter the current price of the car (in lakhs): "))
kms_driven = int(input("Enter the total kilometers driven: "))
fuel_type = input("Enter fuel type (PETROL/DIESEL/CNG): ").strip().upper()
seller_type = input("Enter seller type (DEALER/INDIVIDUAL): ").strip().upper()
transmission = input("Enter transmission type (MANUAL/AUTOMATIC): ").strip().upper()
owner = int(input("Enter number of previous owners (0/1/2/...): "))
year = int(input("Enter the year of purchase: "))

fuel_dict = {'PETROL': 0, 'Diesel': 1, 'CNG': 2}
seller_dict = {'DEALER': 0, 'INDIVIDUAL': 1}
transmission_dict = {'MANUAL': 0, 'AUTOMATIC': 1}

fuel_type = fuel_dict.get(fuel_type, -1)
seller_type = seller_dict.get(seller_type, -1)
transmission = transmission_dict.get(transmission, -1)
age = 2025 - year

if -1 in [fuel_type, seller_type, transmission]:
    print("Invalid input detected. Please enter correct values!")
else:
    features = np.array([[present_price, kms_driven, fuel_type, seller_type, transmission, owner, age]])
    predicted_price = LRmodel.predict(features)[0]  
    print(f"Predicted Resale Price: ₹{predicted_price:.2f} lakhs")

plt.figure(figsize=(8, 5))
plt.bar(['Current Price', 'Predicted Resale Price'], [present_price, predicted_price], color=['blue', 'green'])
plt.ylabel('Price (in lakhs)')
plt.title('Car Price Comparison')
for i, v in enumerate([present_price, predicted_price]):
    plt.text(i, v + 0.5, f'₹{v:.2f}', ha='center')  
plt.show()

X_with_const = sm.add_constant(X)
vif_data = pd.DataFrame()
vif_data['Feature'] = X_with_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
print("\nVIF Values:")
print(vif_data)

X_ols = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_ols).fit()
print("\nOLS Regression Results:")
print(ols_model.summary())

plt.figure(figsize=(8, 6))
sns.regplot(x=X_train['Present_Price'], y=LRmodel.predict(X_train), scatter_kws={'color': 'red'}, line_kws={'color': 'blue'})
plt.xlabel('Present Price')
plt.ylabel('Predicted Price')
plt.title('Regression Plot')
plt.show()

y_train_pred = LRmodel.predict(X_train)
y_test_pred = LRmodel.predict(X_test)
mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
print(f"\nMSE Train: {mse_train:.2f}")
print(f"MSE Test: {mse_test:.2f}")

percentage_diff = ((present_price-predicted_price)/present_price)* 100 
print('Percentage Difference between current price and predicted resale price:',percentage_diff)

KeyboardInterrupt: 